In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path

# Directory containing *_clean.pdb snapshots (from NB03)
SNAPSHOTS_DIR = Path("../snapshots")

# Ligand residue name in the PDB files
LIGAND_RESNAME = "UNK"

# Output directories
OUTPUT_DIR  = Path("./results/pocket")
FIGURES_DIR = Path("./figures/pocket")

# Which PDB to use as the 2D structure template (first match used)
# Set to None to use the first PDB found in SNAPSHOTS_DIR.
BASE_PDB: Path | None = None
# ============================================================

In [ ]:
# Requires: pip install 'mdatools[pocket]'
import pandas as pd
from mdatools.pocket import PocketProfiler
from mdatools.pocket.profiler import build_2d_mol
from mdatools.plotting.pocket_profile import make_snapshot_png

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

clean_pdbs = sorted(SNAPSHOTS_DIR.glob("*_clean.pdb"))
assert clean_pdbs, f"No *_clean.pdb found in {SNAPSHOTS_DIR}"
print(f"Found {len(clean_pdbs)} snapshot PDBs")

In [ ]:
# Build 2D structure from base PDB
base_pdb = BASE_PDB or clean_pdbs[0]
print(f"2D template: {base_pdb.name}")
mol, name2idx = build_2d_mol(base_pdb, ligand_resname=LIGAND_RESNAME)
print(f"  atoms={mol.GetNumAtoms()}, mapped={len(name2idx)}")

In [ ]:
# Compute pocket metrics for all snapshots
profiler = PocketProfiler(ligand_resname=LIGAND_RESNAME)
all_metrics = profiler.profile_batch(clean_pdbs, output_dir=OUTPUT_DIR)
print(f"Profiled {len(all_metrics)} snapshots")

In [ ]:
# Generate overlay PNGs for each snapshot
for pdb_path in clean_pdbs:
    stem    = pdb_path.stem
    metrics = all_metrics[stem]
    png_out = FIGURES_DIR / f"pocket_{stem}.png"
    make_snapshot_png(
        mol, name2idx, metrics,
        out_path=png_out,
        title=f"Pocket environment | {stem}",
    )

print("Done. PNGs written to:", FIGURES_DIR)

In [ ]:
# Styled summary table — most tightly packed atoms across all snapshots
frames = []
for stem, metrics in all_metrics.items():
    df = pd.DataFrame(metrics)
    df["snapshot"] = stem
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

# Show per-atom average d_min across snapshots
pivot = (
    combined.groupby("atom")[["d_min", "d_margin", "hydrophob", "n_NO"]]
    .mean()
    .sort_values("d_min")
    .round(2)
)

pivot.style.background_gradient(subset=["d_min"], cmap="RdYlGn", vmin=3.0, vmax=8.0)

In [ ]:
# Display one PNG inline
from IPython.display import Image as IPyImage
first_png = sorted(FIGURES_DIR.glob("*.png"))[0]
IPyImage(filename=str(first_png), width=900)